# Train Model A - Burnout Classifier

The original Model A (`modelA.pkl`) was 201 MB (XGBoost, 2019 estimators, depth 12, trained on 2.1 million rows with SMOTE); too large to be transpiled to **JavaScript** via **m2cgen** for web deployment (Vercel).

This notebook retrains Model A with a much smaller configuration:
- **Same algorithm**: XGBoost (supported by m2cgen)
- **n_estimators**: 2019 → 200
- **max_depth**: 12 → 5
- **Same preprocessing**: IQR cleaning, `engineer_features_a`, SMOTE

These new smaller configurations are obtained from the stepwise tuning process where Optuna and RSCV are not suitable since their running time (which is around 3-8 hours) is limited by the time constraint.

Output: `models/modelA.pkl` (~1.6 MB, Macro F1 ~0.53; on par with the original, but with much less memory size)

## 1. Imports & Config

In [24]:
import sys
import os
import time
import numpy as np
import pandas as pd
import joblib
import warnings

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from imblearn.over_sampling import SMOTE
import xgboost as xgb

# From the notebooks folder go up one level so you can import src.features
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from src.features import engineer_features_a

In [25]:
DATA_PATH = os.path.join(REPO_ROOT, "datasets", "academic_stress_level.csv") # Get the correct address of academic_stress_level.csv file dataset
OUT_PATH  = os.path.join(REPO_ROOT, "models", "modelA.pkl") # Get the address of correct modelA.pkl

# The 10 base features from the original dataset
FEATURE_COLS = [
    "study_hours_per_day", "sleep_hours", "exam_pressure", "stress_level",
    "financial_stress", "social_support", "anxiety_score", "depression_score",
    "family_expectation", "physical_activity",
]
CLASS_NAMES = {0: "Healthy", 1: "Mildly Burnout", 2: "Burnout"} # Target classes

# Obtained from the stepwise tuning
N_ESTIMATORS  = 200
MAX_DEPTH     = 5
LEARNING_RATE = 0.15

## 2. Load & Preprocess Data

In [26]:
df = pd.read_csv(DATA_PATH) # From DATA_PATH, read the CSV file (which is academic_stress_level.csv) and load it into the dataframe
print(f"Raw dataset: {len(df)} rows")

# Turning the burnout_score column, which is numerical into a brand new categorical column named "burnout_class", which is going to be the column output
def bin_burnout(score):
    if score < 4:   
        return 0
    elif score < 7: 
        return 1
    return 2

# On each row in burnout_score column, apply bin_burnout function
df["burnout_class"] = df["burnout_score"].apply(bin_burnout)
df_model = df[FEATURE_COLS + ["burnout_class"]].dropna() # Obtain the final dataframe 

# IQR cleaning on features only (preserves all class samples)
for col in FEATURE_COLS:
    q1, q3 = df_model[col].quantile(0.25), df_model[col].quantile(0.75)
    iqr = q3 - q1
    if iqr == 0:
        continue
    mask = (df_model[col] >= q1 - 1.5 * iqr) & (df_model[col] <= q3 + 1.5 * iqr)
    df_model = df_model[mask]

print(f"After IQR clean: {len(df_model)} rows")

Raw dataset: 1000000 rows
After IQR clean: 981357 rows


## 3. Feature Engineering

In [27]:
# Apply feature engineering tool to the final model, df_model
df_model = engineer_features_a(df_model)

# Count the number of engineered features, except the burnout_class column since it is the output/target column
all_features = [c for c in df_model.columns if c != "burnout_class"]
print(f"Total features after engineering: {len(all_features)}")

X = df_model[all_features]
y = df_model["burnout_class"]

Total features after engineering: 20


## 4. Train/Test Split + SMOTE

In [28]:
# Splitting the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train_sm, y_train_sm = SMOTE(random_state=42).fit_resample(X_train, y_train) # Applying SMOTE to reduce the effect of severe data imbalance
print(f"After SMOTE: {len(X_train_sm)} training rows")

After SMOTE: 2105241 training rows


## 5. Training

In [29]:
clf = xgb.XGBClassifier( # Training the model and save it into the clf variable
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    learning_rate=LEARNING_RATE,
    objective="multi:softprob",
    num_class=3,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    verbosity=0,
)

clf.fit(X_train_sm, y_train_sm)

,objective,'multi:softprob'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


## 6. Evaluation

In [30]:
y_pred = clf.predict(X_test)
acc  = accuracy_score(y_test, y_pred)
f1m  = f1_score(y_test, y_pred, average="macro")

print("Model A - Evaluation Metrics:")
print(f"Accuracy : {acc:.4f}")
print(f"Macro-F1 : {f1m:.4f}")
print(classification_report(y_test, y_pred, target_names=list(CLASS_NAMES.values())))

Model A - Evaluation Metrics:
Accuracy : 0.8465
Macro-F1 : 0.5279
                precision    recall  f1-score   support

       Healthy       0.98      0.86      0.92    175437
Mildly Burnout       0.38      0.73      0.50     20405
       Burnout       0.09      0.71      0.17       430

      accuracy                           0.85    196272
     macro avg       0.49      0.77      0.53    196272
  weighted avg       0.92      0.85      0.87    196272



## 6b. Cross-Validation

5-fold StratifiedKFold with SMOTE inside each fold (no leakage), confirms the compact model is stable, not overfit to one split.

In [31]:
from sklearn.model_selection import StratifiedKFold, cross_validate
from imblearn.pipeline import Pipeline as ImbPipeline

def _build_clf():
    return xgb.XGBClassifier(
        n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH, learning_rate=LEARNING_RATE,
        objective="multi:softprob", num_class=3, tree_method="hist",
        random_state=42, n_jobs=-1, verbosity=0,
    )

pipe = ImbPipeline([("smote", SMOTE(random_state=42)), ("clf", _build_clf())])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv = cross_validate(pipe, X, y, cv=skf, scoring=["accuracy", "f1_macro"], n_jobs=1)

print("Model A - Cross-Validation (5-fold):")
print(f"CV Accuracy : {cv['test_accuracy'].mean():.4f} +/- {cv['test_accuracy'].std():.4f}")
print(f"CV Macro-F1 : {cv['test_f1_macro'].mean():.4f} +/- {cv['test_f1_macro'].std():.4f}")


Model A - Cross-Validation (5-fold):
CV Accuracy : 0.8464 +/- 0.0006
CV Macro-F1 : 0.5265 +/- 0.0021


## 7. Save the Model

In [32]:
joblib.dump(clf, OUT_PATH)
print(f"Saved model A successfully!")

Saved model A successfully!
